# 01 — Exploratory Data Analysis & Preprocessing

**Project:** Hospital Readmission Predictor  
**Author:** Harshita Adlakha  
**Dataset:** 130 US Hospitals Diabetes Dataset (1999–2008)

This notebook covers:
1. Loading and inspecting the dataset
2. Distribution analysis and outlier detection
3. Missing value analysis
4. Applying the `DataPreprocessor` pipeline
5. Feature engineering with `FeatureEngineer`

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import sys
sys.path.insert(0, '..')

from src.preprocessing import DataPreprocessor
from src.feature_engineering import FeatureEngineer

plt.rcParams['figure.figsize'] = (10, 5)
sns.set_style('whitegrid')
print('Libraries loaded.')

## 1. Load Data

In [ ]:
train = pd.read_csv('../data/train.csv')
test  = pd.read_csv('../data/test.csv')

print(f'Train: {train.shape}  |  Test: {test.shape}')
train.head()

## 2. Basic Info

In [ ]:
train.info()

In [ ]:
train.describe()

## 3. Missing Values

In [ ]:
# Replace sentinel missing values
train_vis = train.replace('?', np.nan)

missing = (train_vis.isnull().sum() / len(train_vis) * 100).sort_values(ascending=False)
missing = missing[missing > 0]

fig, ax = plt.subplots(figsize=(10, 5))
missing.plot(kind='bar', color='steelblue', ax=ax)
ax.axhline(50, color='red', linestyle='--', label='50% threshold')
ax.set_title('Missing Value Rate by Feature')
ax.set_ylabel('% Missing')
ax.legend()
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print(missing)

## 4. Target Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

if 'readmitted' in train.columns:
    train['readmitted'].value_counts().plot(kind='bar', ax=axes[0], color=['#4C72B0','#DD8452','#55A868'])
    axes[0].set_title('Multiclass Target: readmitted')
    axes[0].set_xlabel('Class')
    axes[0].set_ylabel('Count')

if 'readmitted_binary' in train.columns:
    train['readmitted_binary'].value_counts().plot(kind='bar', ax=axes[1], color=['#4C72B0','#DD8452'])
    axes[1].set_title('Binary Target: readmitted_binary')
    axes[1].set_xlabel('Class (0=No, 1=<30 days)')
    axes[1].set_ylabel('Count')

plt.tight_layout()
plt.show()

## 5. Numeric Feature Distributions

In [ ]:
numeric_cols = ['time_in_hospital', 'num_lab_procedures', 'num_procedures',
                'num_medications', 'number_diagnoses', 'number_outpatient',
                'number_emergency', 'number_inpatient']

numeric_cols = [c for c in numeric_cols if c in train.columns]

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    axes[i].hist(train[col].dropna(), bins=30, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_ylabel('Frequency')

plt.suptitle('Numeric Feature Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 6. Apply DataPreprocessor

In [ ]:
TARGET_COLS = [c for c in ('readmitted', 'readmitted_binary') if c in train.columns]

X_train_raw = train.drop(columns=TARGET_COLS)
y_bin  = train['readmitted_binary'] if 'readmitted_binary' in train.columns else None
y_mul  = train['readmitted'] if 'readmitted' in train.columns else None

preprocessor = DataPreprocessor(missing_threshold=0.5)
preprocessor.fit(X_train_raw)
X_proc = preprocessor.transform(X_train_raw)

print(f'Shape after preprocessing: {X_proc.shape}')
print(f'Remaining nulls: {X_proc.isnull().sum().sum()}')
X_proc.head()

## 7. Apply FeatureEngineer

In [ ]:
engineer = FeatureEngineer()
engineer.fit(X_proc)
X_eng = engineer.transform(X_proc)

print(f'Shape after feature engineering: {X_eng.shape}')

new_features = ['recurrency', 'medication_change_ratio', 'number_prior_visits',
                'lab_tests_per_med', 'patient_severity', 'age_times_medications']

fig, axes = plt.subplots(2, 3, figsize=(14, 7))
axes = axes.flatten()

for i, feat in enumerate(new_features):
    if feat in X_eng.columns:
        axes[i].hist(X_eng[feat].dropna(), bins=25, color='#55A868', edgecolor='white')
        axes[i].set_title(feat)

plt.suptitle('Engineered Feature Distributions', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## 8. Correlation Heatmap

In [ ]:
corr = X_eng.select_dtypes(include=np.number).corr()

fig, ax = plt.subplots(figsize=(14, 12))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=False, cmap='coolwarm', center=0,
            square=True, linewidths=0.5, ax=ax)
ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

## Summary

- Dataset has **29 raw features** across demographics, clinical, and medication categories.
- `weight` and `payer_code` are dropped due to >50% missing values.
- ICD-9 diagnosis codes are mapped to 8 clinical disease categories.
- 6 engineered features capture clinical relationships not present in raw data.
- **`recurrency`** (prior inpatient visits) is expected to be the top predictor — see classification notebooks.